# obj2 · Meio de mosto sintético MS300 — definição e crescimento solo

**TASK 3.1** — comunidades microbianas para vinho de baixo teor alcoólico

A composição do MS300 é traduzida em bounds de captação pela relação de Sauer
para cultura em batch, e os quatro modelos curados no `obj1a` e no `obj1b` são
validados em crescimento solo.

**A equação:**

$$q_S = \mu \cdot \frac{\Delta S}{X(t_2)}$$

Em batch, dS/dt = −q_S·X e dX/dt = µ·X, logo dS/dX = −q_S/µ e ΔS = (q_S/µ)·ΔX.
Com X(t₁) ≪ X(t₂) fica a forma acima. O tempo não entra: quem ocupa o lugar de
`t` são µ e a biomassa final, que são grandezas medidas.

A versão usada no projeto anterior era q_S ≈ S₀/(X̄·t), com X̄ = 0.5 gDW/L e
t = 24 h escolhidos sem fonte. Isso dá um denominador de 12 gDW·h/L contra os 29
que uma fermentação vínica real tem.

**Os dois parâmetros**, e são só dois: µ = 0.2 h⁻¹ e X(t₂) = 5.8 gDW/L, ambos de
Varela, Pizarro & Agosin (2004) em mosto sintético a 300 mg N/L, que é o azoto do
MS300. Um só par para os quatro modelos, porque em q_S = µ·ΔS/X(t₂) o par
descreve o regime da fermentação e não o organismo.

**A regra do meio:** aplica-se a receita do MS300 e mais nada. O que a receita
não tem não se acrescenta por precaução; mede-se o crescimento só com o meio, e
o que falta a cada organismo sai do rastreio de auxotrofias da secção 5 e entra
depois, declarado, com o rastreio como justificação.

Tudo o que é receita, mapa de trocas e conversão está em `scripts/meio_mosto.py`.
Este notebook só orquestra e verifica.

**Oxigénio:** segue o que ficou decidido no `obj1b`. A *S. cerevisiae*, a
*T. delbrueckii* e a *L. thermotolerans* correm em anaerobiose estrita por
`CU.modo_anaerobio`; a *M. pulcherrima* corre no O₂ declarado em
`CU.Q_O2_DECLARADO`, porque não tem variante anaeróbia.

## 1 · Preparação

In [1]:
import sys, warnings, os
from pathlib import Path
import cobra, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220, "display.max_colwidth", 60)

def raiz(p=None):
    p = (Path(p) if p else Path.cwd()).resolve()
    for c in (p, *p.parents):
        if (c / "models").is_dir() and (c / "scripts").is_dir():
            return c
    raise RuntimeError("raiz do projecto nao encontrada")

ROOT = raiz()
sys.path.insert(0, str(ROOT / "scripts"))
import meio_mosto as MM
import curadoria_minima as CU

OUT = ROOT / "results" / "obj2"
OUT.mkdir(parents=True, exist_ok=True)

_lic = os.path.expanduser("~/gurobi.lic")
if os.path.exists(_lic):
    os.environ["GRB_LICENSE_FILE"] = _lic

FICHEIRO = {"Sc": "yeast-GEM_curado.xml", "Td": "model_delbrueckii_curado.xml",
            "Mp": "model_pulcherrima_curado.xml", "Lt": "model_thermotolerans_curado.xml"}
ESPECIE = {"Sc": "Saccharomyces cerevisiae", "Td": "Torulaspora delbrueckii",
           "Mp": "Metschnikowia pulcherrima", "Lt": "Lachancea thermotolerans"}
BIOMASSA = {"Sc": "r_2111", "Td": "BIOMASS", "Mp": "BIOMASS", "Lt": "BIOMASS"}
ETANOL = {"Sc": "r_1761", "Td": "UF03263_E", "Mp": "UF03263_E", "Lt": "UF03263_E"}
ANA_NS = ["NS2b", "NS3", "NS1"]          # variante anaerobia do obj1b

MODELOS, REGIME = {}, {}
for cod, f in FICHEIRO.items():
    m = cobra.io.read_sbml_model(str(ROOT / "models" / "curados" / f))
    if cod == "Mp":
        q = CU.Q_O2_DECLARADO["Mp"]["microaerobiose"]
        REGIME[cod] = f"microaerobiose, O2 = {q}"
    elif cod == "Sc":
        CU.modo_anaerobio(m, "Sc"); q = 0.0; REGIME[cod] = "anaerobiose estrita"
    else:
        CU.modo_anaerobio(m, cod, ids=ANA_NS); q = 0.0
        REGIME[cod] = "anaerobiose estrita"
    MODELOS[cod] = (m, q)
    print(f"{cod} ({ESPECIE[cod]}): {len(m.reactions)} reaccoes | {REGIME[cod]}")

Sc (Saccharomyces cerevisiae): 4102 reaccoes | anaerobiose estrita


Td (Torulaspora delbrueckii): 1971 reaccoes | anaerobiose estrita


Mp (Metschnikowia pulcherrima): 2131 reaccoes | microaerobiose, O2 = 0.5


Lt (Lachancea thermotolerans): 2139 reaccoes | anaerobiose estrita


## 2 · A equação, verificada à mão

Antes de qualquer simulação: a função tem de reproduzir um cálculo feito à mão, e
o bound tem de multiplicar de volta para a concentração da receita.

A verificação de trás para a frente é a que responde à pergunta "estes bounds não
são pequenos de mais?". Um bound de 0.074 mmol/gDW/h para a arginina não é pequeno
nem grande: é a taxa que esvazia o mosto exatamente quando a biomassa chega aos
5.8 gDW/L.

In [2]:
comp = MM.composicao_mmol_L()

# arginina, a conta inteira
C_arg = MM.RECEITA_MG_L["arginine"]
manual = -(MM.MU_REF * (C_arg / MM.MW["arginine"])) / MM.X_REF
calculado = MM.calcular_bound(comp["arginine"])
assert abs(manual - calculado) < 1e-12
print(f"arginina: {C_arg:.0f} mg/L / {MM.MW['arginine']} g/mol = "
      f"{comp['arginine']:.4f} mmol/L")
print(f"          x ({MM.MU_REF} / {MM.X_REF}) = {calculado:.5f} mmol/gDW/h")
print(f"          verifica de volta: {MM.verificar_bound(calculado):.4f} mmol/L\n")

# glucose, o outro extremo
b_glc = MM.calcular_bound(comp["glucose"])
print(f"glucose : {comp['glucose']:.1f} mmol/L -> {b_glc:.4f} mmol/gDW/h")
print(f"          verifica de volta: {MM.verificar_bound(b_glc):.1f} mmol/L")
print(f"          hexose total disponivel: {2*abs(b_glc):.2f} mmol/gDW/h\n")

# a receita reproduz o '300' do MS300?
print("YAN nas duas convencoes:", MM.yan())
print("  a enologica (1 N por aminoacido) e a que baptiza o MS300.")
print("  a biomassa consome atomos, e a arginina tem 4.")

arginina: 374 mg/L / 174.2 g/mol = 2.1470 mmol/L
          x (0.2 / 5.8) = -0.07403 mmol/gDW/h
          verifica de volta: 2.1470 mmol/L

glucose : 555.1 mmol/L -> -19.1401 mmol/gDW/h
          verifica de volta: 555.1 mmol/L
          hexose total disponivel: 38.28 mmol/gDW/h

YAN nas duas convencoes: {'YAN_alfa_amino_mgN_L': 300.7, 'N_em_atomos_mgN_L': 459.2}
  a enologica (1 N por aminoacido) e a que baptiza o MS300.
  a biomassa consome atomos, e a arginina tem 4.


## 3 · Composição e mapeamento das trocas

A receita vem de Evers et al. 2023, *Foods* 12(5):972, secção 2.2, transcrita e
auditada em `ms300_medium_fixed.py`. Os sais e o Tween 80 dissociam-se antes de
virar bound: o Tween 80 é monooleato de polioxietileno(20)-sorbitano e traz
1 mol de oleato por mol nominal, o que multiplica por 22 o oleato disponível.

O mapa de trocas resolve-se por uma tabela de sinónimos, e não por um mapa
escrito à mão por modelo, porque as duas famílias de ficheiros nomeiam de maneira
diferente: o yeast-GEM escreve `L-glycine` e `L-arginine`, o CarveFungi escreve
`glycine zwitterion` e `L-argininium`. É a tabela `MM.SINONIMOS` o artefacto a
auditar.

In [3]:
tab = MM.tabela_do_meio(MODELOS["Sc"][0])
print(f"{len(comp)} especies na composicao, apos dissociacao dos sais\n")
print(tab.to_string(index=False))
tab.to_csv(OUT / "composicao_e_bounds_Sc.csv", index=False)

45 especies na composicao, apos dissociacao dos sais

      especie   mmol_L     troca  eficiencia     bound  verifica_mmol_L  N_atomos
      glucose 555.0622    r_1714         1.0 -19.14007         555.0622         0
     fructose 555.0622    r_1709         1.0 -19.14007         555.0622         0
       malate  44.7461    r_1552         1.0  -1.54297          44.7461         0
      citrate  31.2305    r_1687         1.0  -1.07691          31.2305         0
     chloride  14.1349    r_4593         1.0  -0.48741          14.1349         0
    potassium  11.2556    r_2020         1.0  -0.38813          11.2556         0
     ammonium   8.6049    r_1654         1.0  -0.29672           8.6049         1
    phosphate   5.5111    r_2005         1.0  -0.19004           5.5111         0
      sulfate   3.9252    r_2060         1.0  -0.13535           3.9252         0
    glutamine   3.4556    r_1891         1.0  -0.11916           3.4556         2
       sodium   3.4223    r_2049         1.0

In [4]:
MAPAS = {cod: MM.construir_mapa(m) for cod, (m, _) in MODELOS.items()}
d_mapa = pd.DataFrame(MAPAS)
print(d_mapa.fillna("(ausente)").to_string())
d_mapa.to_csv(OUT / "mapa_trocas.csv")

print("\nespecies da receita ausentes de cada modelo:")
for cod in MODELOS:
    aus = [k for k in comp if MAPAS[cod].get(k) is None]
    print(f"  {cod} ({len(aus)}): {aus}")

                              Sc         Td         Mp         Lt
glucose                   r_1714  UF03288_E  UF03288_E  UF03288_E
fructose                  r_1709  UF03275_E  UF03275_E  UF03275_E
malate                    r_1552  UF03510_E  UF03510_E  UF03510_E
citrate                   r_1687  UF02577_E  UF02577_E  UF02577_E
potassium                 r_2020  (ausente)  (ausente)  (ausente)
phosphate                 r_2005  UF02765_E  UF02765_E  UF02765_E
sulfate                   r_2060  UF03456_E  UF03456_E  UF03456_E
magnesium                 r_4597  (ausente)  (ausente)  (ausente)
calcium                   r_4600  (ausente)  (ausente)  (ausente)
chloride                  r_4593  (ausente)  (ausente)  (ausente)
sodium                    r_2049  (ausente)  (ausente)  (ausente)
ammonium                  r_1654  UF03376_E  UF03376_E  UF03376_E
manganese                 r_4595  (ausente)  (ausente)  (ausente)
zinc                      r_4596  (ausente)  (ausente)  (ausente)
copper    

As ausências não são acidentes e cada uma tem uma leitura:

| ausente | onde | decisão |
|---|---|---|
| iodeto, cobalto, borato, molibdato | os quatro modelos | oligoelementos que nenhuma reconstrução tem. Não entram, e nenhum é indispensável nos testes T8 |
| K, Mg, Ca, Cl, Na, Mn, Zn, Cu | os três CarveFungi | não têm trocas minerais separadas: têm o pseudo-metabolito `ash 1 g`. É a limitação declarada no `obj1b` |
| ferro | **não está na receita** | o MS300 não tem sal de ferro, e o ferro é indispensável nos quatro modelos (T8). Entra na camada basal, declarado, e não derivado da receita |

## 4 · As camadas

`aplicar_meio_mosto` aplica o meio, e só o meio:

1. fecha toda a captação, deixando a secreção livre
2. **basais** sem tecto: água, protão e CO₂. Solvente, pH e produto, não são
   nutrientes e não limitam nada
3. **`ash`** nos modelos CarveFungi, derivado da receita: o pseudo-metabolito
   `ash 1 g` tem unidade de 1 g, por isso o seu ΔS é a massa de sais minerais do
   MS300, 1.867 g/L, que dá um bound de 0.0644
4. **composição**: tudo o resto, pela equação
5. oxigénio no regime de cada organismo

**Nada que não esteja na receita entra aqui.** Nem ferro, nem esteróis extra,
nem tectos de conveniência para as vitaminas. Acrescentar por precaução é
responder a uma pergunta que ainda não foi feita, e esconde o resultado que
interessa. O que falta a cada organismo mede-se na secção 5.

In [5]:
cod = "Sc"
m, q_o2 = MODELOS[cod]
with m:
    ap, aus = MM.aplicar_meio_mosto(m, mapa=MAPAS[cod], q_o2=q_o2, verbose=True)
    abertas = [(r.id, (next(iter(r.metabolites)).name or "")[:26], round(r.lower_bound, 6))
               for r in m.exchanges if r.lower_bound < 0]
d = pd.DataFrame(sorted(abertas, key=lambda x: x[2]), columns=["troca", "metabolito", "lower_bound"])
print(f"\n{len(d)} trocas abertas a captacao no {cod}")
print(d.to_string(index=False))
print(f"\nash: {MM.ash_mmol_L():.3f} g/L de sais -> bound {MM.calcular_bound(MM.ash_mmol_L()):.5f}")

  aplicados: 41 | ausentes do modelo: 4 -> ['iodide', 'cobalt', 'borate', 'molybdate']
  O2 = 0.0 mmol/gDW/h



44 trocas abertas a captacao no Sc
 troca       metabolito  lower_bound
r_1672   carbon dioxide -1000.000000
r_1832               H+ -1000.000000
r_2100              H2O -1000.000000
r_1709       D-fructose   -19.140075
r_1714        D-glucose   -19.140075
r_1552       (S)-malate    -1.542968
r_1687          citrate    -1.076913
r_4593         chloride    -0.487409
r_2020        potassium    -0.388125
r_1654         ammonium    -0.296720
r_2005        phosphate    -0.190037
r_2060         sulphate    -0.135351
r_1891      L-glutamine    -0.119158
r_2049           sodium    -0.118011
r_1879       L-arginine    -0.074033
r_1873        L-alanine    -0.056123
r_4600           Ca(2+)    -0.036465
r_4597           Mg(2+)    -0.034977
r_1912     L-tryptophan    -0.030223
r_1889      L-glutamate    -0.028124
r_1906         L-serine    -0.025922
r_1911      L-threonine    -0.022000
r_1914         L-valine    -0.013246
r_1899        L-leucine    -0.012619
r_1881      L-aspartate    -0.011658
r_

## 5 · Auxotrofias face ao mosto

A pergunta certa antes de acrescentar seja o que for: **com o meio exatamente
como está definido, quem cresce e o que falta a quem não cresce.**

`rastrear_auxotrofias` faz três passos. Aplica a receita e mede o µ. Se não
houver crescimento, abre todas as trocas que não são do meio a 0.02 e volta a
medir: se nem assim crescer, o bloqueio não é nutricional. Se crescer, fecha-as
uma a uma e fica com as que, ao fechar, matam o crescimento.

Esse conjunto é minimal, não mínimo: depende da ordem por que se fecham as
trocas e pode ficar com um membro esquisito quando havia alternativas. Por isso
corre-se também `substitutos`, que para cada membro procura quem o pode
substituir. É a lista de substitutos que dá a leitura: "um esterol esterificado
ou um C16:1" é uma conclusão, "éster de colesterol" é um acaso da ordem.

O oxigénio fica de fora do rastreio: é regime, não é meio.

In [6]:
AUX = {}
for cod in MODELOS:
    m, q_o2 = MODELOS[cod]
    print(f"\n{'='*70}\n{cod} — {ESPECIE[cod]}  [{REGIME[cod]}]\n{'='*70}")
    mu_meio, mu_tudo, ess = MM.rastrear_auxotrofias(m, mapa=MAPAS[cod], q_o2=q_o2)
    AUX[cod] = {"mu_meio": mu_meio, "mu_tudo_aberto": mu_tudo, "essenciais": ess}
    if ess:
        subs = MM.substitutos(m, ess, mapa=MAPAS[cod], q_o2=q_o2)
        AUX[cod]["substitutos"] = subs
        for rid, d in subs.items():
            print(f"     {rid} ({d['nome']}): {d['n_substitutos']} substitutos")
            for sid, snome in d["substitutos"]:
                print(f"        {sid:12s} {snome}")


Sc — Saccharomyces cerevisiae  [anaerobiose estrita]


  mu so com o meio = 0.0000 | com tudo aberto a 0.02 = 0.0894
  conjunto minimo que desbloqueia (2):
     r_1861       iron(2+)                         fluxo 0.0
     r_1994       palmitoleate                     fluxo 0.00066


     r_1861 (iron(2+)): 0 substitutos
     r_1994 (palmitoleate): 0 substitutos

Td — Torulaspora delbrueckii  [anaerobiose estrita]


  mu so com o meio = 0.0000 | com tudo aberto a 0.02 = 0.1034
  conjunto minimo que desbloqueia (1):
     UF03214_E    cholesterol ester                fluxo 0.011183


     UF03214_E (cholesterol ester): 5 substitutos
        UF02972_E    (9Z)-hexadecenoate
        UF03015_E    lanosterol ester
        UF03013_E    episterol ester
        UF03014_E    fecosterol ester
        UF03011_E    zymosterol ester

Mp — Metschnikowia pulcherrima  [microaerobiose, O2 = 0.5]


  mu so com o meio = 0.0000 | com tudo aberto a 0.02 = 0.3895
  conjunto minimo que desbloqueia (1):
     UF03268_E    iron(2+)                         fluxo 3.7e-05


     UF03268_E (iron(2+)): 0 substitutos

Lt — Lachancea thermotolerans  [anaerobiose estrita]


  mu so com o meio = 0.0000 | com tudo aberto a 0.02 = 0.3696
  conjunto minimo que desbloqueia (1):
     UF03214_E    cholesterol ester                fluxo 0.011183


     UF03214_E (cholesterol ester): 5 substitutos
        UF02972_E    (9Z)-hexadecenoate
        UF03011_E    zymosterol ester
        UF03013_E    episterol ester
        UF03015_E    lanosterol ester
        UF03012_E    ergosterol ester


### 5.1 · O que entra como suplemento declarado, e porquê

Duas listas, com justificações diferentes. Não se misturam, porque a razão de
cada uma é outra.

**`SUPLEMENTOS_AUSENTES` — não estão na receita.** Entram porque o rastreio
mostrou que sem eles nenhum modelo cresce, e mostrou também que nada mais falta.

| falta | a quem | substitutos | leitura |
|---|---|---|---|
| **ferro** | Sc e Mp | nenhum | nem a secção 2.2 do Evers nem o MS300 clássico do Bely têm sal de ferro, e o ferro é indispensável (T8). Na Td e na Lt deixa de ser essencial porque a edição NS2b tira os citocromos da biomassa anaeróbia |
| **palmitoleato (C16:1)** | Sc, Td e Lt | na Td e na Lt, qualquer éster de esterol | o Tween 80 traz C18:1 e mais nada. A fração lipídica das equações de biomassa pede C16:1 em proporção fixa |

**`SUPLEMENTOS_PROTOCOLO` — estão na receita**, e saem da camada derivada da
composição na mesma. A razão é outra e tem de ficar escrita: o tecto do mosto
mata o modelo, e a culpa é da equação de biomassa e não do mosto.

A escada medida no yeast-GEM em anaerobiose, com tecto de 0.02 no que se liberta:

| | µ | |
|---|---|---|
| só o meio, mais os ausentes | 0.0071 | não é o que se aplica |
| + ergosterol e oleato | 0.0923 | não é o que se aplica |
| + nicotinato | **0.1654** | **← é esta a configuração aplicada** |
| + piridoxina e as outras vitaminas | 0.1689 | não se aplica: ver a nota da piridoxina |

**Esta tabela é a medição que justifica a decisão, não um resultado do
notebook.** Os 0.0071 da primeira linha são o que a *S. cerevisiae* dava antes
desta secção existir. O que o notebook corre a partir daqui é a terceira linha, e
é o µ = 0.1654 que aparece nas secções 6 e 7.

O esterol que a biomassa do yeast-GEM consome equivale a **1.48 % do peso seco**;
o MS300 fornece **0.052 %** a X(t₂) = 5.8, o que dá um fator de 30. Uma levedura
anaeróbia limitada em esterol baixa a fração de esterol da membrana, e a equação
de biomassa não pode. É exatamente por isso que o protocolo anaeróbio do próprio
yeast-GEM (`anaerobicModel.m`) abre esteróis, ácidos gordos insaturados,
nicotinato e pantotenato sem tecto derivado do meio.

**Consequência a declarar: o eixo do esterol não está modelado.** Qualquer
conclusão deste pipeline sobre limitação por esterol está a ser lida por cima de
um tecto declarado, não medida.

A **piridoxina fica de fora** das duas listas. Sem tecto vai a 1.7 mmol/gDW/h, o
equivalente a 8.3 g/L no mosto: deixa de ser vitamina e passa a ser fonte de
carbono. O tecto da receita tem de ficar.

## 6 · Crescimento solo, e o que limita

Três cenários. No mosto sintético a composição é fixa por definição, por isso o
que varia entre cenários é o **par de parâmetros**, que é onde está a incerteza:

| cenário | µ | X(t₂) | de onde vem |
|---|---|---|---|
| mínimo | 0.05 | 1.5 | fermentação lenta a 50 mg N/L (Varela et al. 2004) |
| central | 0.20 | 5.8 | fermentação normal a 300 mg N/L (Varela et al. 2004) |
| máximo | 0.25 | 8.0 | MS300 a 324 mg N/L (Brou et al., OENO One) |

A coluna que interessa não é o µ: é **o que prende**. Um meio de mosto tem de ser
limitado em azoto, que é o que faz as fermentações vínicas pararem.

In [7]:
from cobra.flux_analysis import pfba

# As duas listas da seccao 5.1, com justificacoes diferentes.
SUPLEMENTOS_DECLARADOS = MM.SUPLEMENTOS
print("ausentes da receita :", MM.SUPLEMENTOS_AUSENTES)
print("fora do tecto do meio:", MM.SUPLEMENTOS_PROTOCOLO)
print("tecto:", MM.Q_SUPLEMENTO, "mmol/gDW/h")

def corre(cod, mu, x_final):
    m, q_o2 = MODELOS[cod]
    with m:
        ap, aus = MM.aplicar_meio_mosto(m, mapa=MAPAS[cod], mu=mu, x_final=x_final,
                                        q_o2=q_o2, suplementos=SUPLEMENTOS_DECLARADOS,
                                        verbose=False)
        v = m.slim_optimize()
        v = 0.0 if v != v else v
        if v < 1e-9:
            return {"mu": 0.0, "cresce": False}
        s = pfba(m)
        hexo = -sum(s.fluxes[MAPAS[cod][k]] for k in ("glucose", "fructose")
                    if MAPAS[cod].get(k))
        etoh = s.fluxes.get(ETANOL[cod], 0.0)
        Cin = sum(-s.fluxes[r.id] * (next(iter(r.metabolites)).elements or {}).get("C", 0)
                  for r in m.exchanges if s.fluxes.get(r.id, 0.0) < -1e-9)
        Cout = sum(s.fluxes[r.id] * (next(iter(r.metabolites)).elements or {}).get("C", 0)
                   for r in m.exchanges if s.fluxes.get(r.id, 0.0) > 1e-9)
        Cbio = Cin - Cout
        prendem = [k for k, (rid, b) in ap.items()
                   if abs(b) > 1e-9 and abs(s.fluxes[rid] - b) < 1e-7]
        return {"mu": round(v, 4), "cresce": True,
                "hexose": round(hexo, 2), "etanol": round(etoh, 2),
                "etanol_por_hexose": round(etoh / hexo, 3) if hexo > 1e-9 else None,
                "C_biomassa_pct": round(Cbio / Cin * 100, 1) if Cin > 1e-9 else None,
                "mmolC_por_gDW": round(Cbio / v, 1),
                "n_prendem": len(prendem), "prendem": ", ".join(prendem[:6])}

linhas = []
for nome, par in MM.CENARIOS.items():
    for cod in MODELOS:
        linhas.append({"cenario": nome, "codigo": cod, **par, **corre(cod, par["mu"], par["x_final"])})
d_solo = pd.DataFrame(linhas)
print(d_solo.to_string(index=False))
d_solo.to_csv(OUT / "crescimento_solo.csv", index=False)

ausentes da receita : ('iron', 'palmitoleate')
fora do tecto do meio: ('ergosterol', 'oleate', 'nicotinate')
tecto: 0.02 mmol/gDW/h


cenario codigo     mu  x_final  cresce  hexose  etanol  etanol_por_hexose  C_biomassa_pct  mmolC_por_gDW  n_prendem                                                           prendem
 minimo     Sc 0.1599      1.5    True    6.85   12.26              1.789            13.6           41.1         19 ammonium, pyridoxine, tyrosine, tryptophan, isoleucine, aspartate
 minimo     Td 0.1774      1.5    True    8.70   15.42              1.772            12.0           37.7         20    ammonium, pyridoxine, biotin, tyrosine, tryptophan, isoleucine
 minimo     Mp 0.1775      1.5    True    6.91   11.97              1.732            15.0           37.7         21    ammonium, pyridoxine, biotin, tyrosine, tryptophan, isoleucine
 minimo     Lt 0.1774      1.5    True    8.69   13.64              1.570            12.1           37.7         20    ammonium, pyridoxine, biotin, tyrosine, tryptophan, isoleucine
central     Sc 0.1654      5.8    True    7.08   12.66              1.788            13.6 

In [8]:
# o que prende, por extenso, no cenario central
print("cenario central — compostos no limite do bound\n")
for cod in MODELOS:
    m, q_o2 = MODELOS[cod]
    with m:
        ap, _ = MM.aplicar_meio_mosto(m, mapa=MAPAS[cod], q_o2=q_o2,
                                      suplementos=SUPLEMENTOS_DECLARADOS, verbose=False)
        s = pfba(m)
        prendem = [(k, round(b, 5)) for k, (rid, b) in ap.items()
                   if abs(b) > 1e-9 and abs(s.fluxes[rid] - b) < 1e-7]
    n = [k for k, _ in prendem if k in MM.N_POR_MOLECULA]
    outros = [k for k, _ in prendem if k not in MM.N_POR_MOLECULA]
    print(f"{cod}: {len(prendem)} no limite")
    print(f"   fontes de azoto ({len(n)}): {', '.join(n)}")
    print(f"   outros ({len(outros)}): {', '.join(outros)}")

cenario central — compostos no limite do bound



Sc: 19 no limite
   fontes de azoto (18): ammonium, tyrosine, tryptophan, isoleucine, aspartate, glutamate, arginine, leucine, threonine, glutamine, alanine, valine, methionine, phenylalanine, serine, histidine, lysine, cysteine
   outros (1): pyridoxine


Td: 20 no limite
   fontes de azoto (18): ammonium, tyrosine, tryptophan, isoleucine, aspartate, glutamate, arginine, leucine, threonine, glycine, glutamine, alanine, valine, methionine, phenylalanine, serine, histidine, lysine
   outros (2): pyridoxine, biotin


Mp: 21 no limite
   fontes de azoto (19): ammonium, tyrosine, tryptophan, isoleucine, aspartate, glutamate, arginine, leucine, threonine, glycine, glutamine, alanine, valine, methionine, phenylalanine, serine, histidine, lysine, cysteine
   outros (2): pyridoxine, biotin


Lt: 20 no limite
   fontes de azoto (18): ammonium, tyrosine, tryptophan, isoleucine, aspartate, glutamate, arginine, leucine, threonine, glycine, glutamine, alanine, valine, methionine, phenylalanine, serine, histidine, lysine
   outros (2): pyridoxine, biotin


### 6.1 · Sensibilidade ao que prende

Com os suplementos declarados aplicados, o que sobra a prender. Relaxa-se um
grupo de cada vez por um fator de dez sobre o bound da receita, com um piso de
0.02 para os que a receita deixa quase a zero, e vê-se quanto sobe o µ.

Os esteróis já não aparecem aqui porque saíram para a camada de suplementos na
secção 5.1. O que fica é o que o mosto realmente limita.

In [9]:
GRUPOS = {
    "vitaminas da receita": ("myo_inositol", "pantothenate", "pyridoxine",
                             "thiamine", "biotin"),
    "azoto": tuple(k for k in MM.N_POR_MOLECULA),
    "acucares": ("glucose", "fructose"),
}

linhas = []
for cod in MODELOS:
    m, q_o2 = MODELOS[cod]
    base = corre(cod, MM.MU_REF, MM.X_REF)["mu"]
    linha = {"codigo": cod, "mu_receita": base}
    for nome, grupo in GRUPOS.items():
        with m:
            MM.aplicar_meio_mosto(m, mapa=MAPAS[cod], q_o2=q_o2,
                                  suplementos=SUPLEMENTOS_DECLARADOS, verbose=False)
            for k in grupo:
                rid = MAPAS[cod].get(k)
                if not rid:
                    continue
                r = m.reactions.get_by_id(rid)
                # RELAXAR, nunca apertar: x10 sobre o bound da receita, com um
                # piso de 0.02 para os que a receita deixa quase a zero.
                r.lower_bound = -max(abs(r.lower_bound) * 10, 0.02)
            v = m.slim_optimize(); v = 0.0 if v != v else v
        linha[nome] = round(v, 4)
    linhas.append(linha)
d_sens = pd.DataFrame(linhas)
print(d_sens.to_string(index=False))
d_sens.to_csv(OUT / "sensibilidade.csv", index=False)
print("\nA coluna que mais sobe e a que domina. Nao se corrige aqui: declara-se.")

codigo  mu_receita  vitaminas da receita  azoto  acucares
    Sc      0.1654                0.1689 0.2096    0.1654
    Td      0.1835                0.1874 0.3266    0.1835
    Mp      0.1836                0.1875 0.8667    0.1836
    Lt      0.1835                0.1874 0.3266    0.1835

A coluna que mais sobe e a que domina. Nao se corrige aqui: declara-se.


## 7 · Valores canónicos para jusante

In [10]:
# Contrato de sincronizacao manual: estes quatro valores sao os que o obj3
# (SMETANA) e o obj4 (comunidade) usam como referencia de crescimento solo.
# Cenario central, meio MS300, regime de O2 do obj1b.
CANONICOS = {cod: corre(cod, MM.MU_REF, MM.X_REF)["mu"] for cod in MODELOS}
print("mu solo, cenario central (mu = 0.20, X = 5.8):")
for cod, v in CANONICOS.items():
    print(f"   {cod} ({ESPECIE[cod]:26s}) {v:.4f} h-1   [{REGIME[cod]}]")
pd.Series(CANONICOS, name="mu_solo").to_csv(OUT / "mu_solo_canonico.csv")

mu solo, cenario central (mu = 0.20, X = 5.8):
   Sc (Saccharomyces cerevisiae  ) 0.1654 h-1   [anaerobiose estrita]
   Td (Torulaspora delbrueckii   ) 0.1835 h-1   [anaerobiose estrita]
   Mp (Metschnikowia pulcherrima ) 0.1836 h-1   [microaerobiose, O2 = 0.5]
   Lt (Lachancea thermotolerans  ) 0.1835 h-1   [anaerobiose estrita]


## 8 · Interpretação e limitações

*A confirmar com o Gurobi; os valores vieram do GLPK.*

**A regra deste notebook:** o meio é o meio. Aplica-se a receita do MS300 e mais
nada, mede-se, e o que falta identifica-se pelo rastreio da secção 5 em vez de
ser acrescentado por precaução. Entraram cinco compostos como suplemento
declarado, dois por ausência da receita e três por decisão de protocolo, e as
duas razões estão separadas na secção 5.1.

**Os quatro modelos ficam limitados em azoto**, com 18 ou 19 fontes de N no
limite do bound. É a assinatura da fermentação vínica e o que faz as fermentações
pararem. Saiu da conversão, sem nada ajustado.

**O µ solo fica entre 0.165 e 0.184 h⁻¹**, contra 0.2 h⁻¹ medido por Varela et al.
em mosto sintético. Os quatro no mesmo intervalo, e a *S. cerevisiae* já não é a
mais lenta, que era o que acontecia antes da decisão da secção 5.1.

**O rendimento em etanol fica entre 1.58 e 1.79 mol/mol**, ou seja 79 a 90 % do
máximo teórico, contra os 79 % que o meio de trabalho genérico do `obj1a` dava.

**Limitações a declarar:**

- **O eixo do esterol não está modelado.** O ergosterol e o oleato saíram da
  camada derivada da composição, com o tecto medido de 0.02. Qualquer conclusão
  deste pipeline sobre limitação por esterol está a ser lida por cima desse
  tecto. A correção certa é uma equação de biomassa anaeróbia, com a fração de
  esterol e de UFA baixada, e é a edição simétrica da NS3 que já se fez do lado
  dos CarveFungi. Deytieux et al. 2005 mediram a exigência de ergosterol por
  estirpe em fermentação alcoólica e é por aí que se começa.
- **O MS300 não tem ferro nem C16:1.** Os dois entram como suplemento, com o
  rastreio da secção 5 como justificação. Quem quiser rigor no eixo do ferro tem
  de lhe pôr um valor de mosto real, 1 a 5 mg/L, e dizê-lo.
- **A piridoxina e a biotina continuam a prender** e ficam assim. São o mesmo
  tipo de problema que o esterol, fração fixa de cofactores contra o que o mosto
  traz, mas de tamanho pequeno: levantar-lhes o tecto vale 0.003 no µ.
- **O eixo mineral não é comparável entre as duas famílias.** O yeast-GEM tem
  trocas separadas para K, Mg, Ca, Mn, Zn e Cu; os três CarveFungi têm `ash 1 g`
  sem fórmula, derivado da receita como 1.867 g/L de sais.
- **Iodeto, cobalto, borato e molibdato não existem em nenhum dos quatro
  modelos.** Nenhum é indispensável nos testes T8, mas fica declarado.
- **A prolina está fora da receita de propósito**, por não constar da secção 2.2
  do Evers e por não ser assimilável em anaerobiose: a prolina oxidase é
  mitocondrial e depende de O₂.
- **Um só par (µ, X) para os quatro modelos**, e o par é entrada e saída ao mesmo
  tempo. Está tratado como entrada declarada, e é por isso que os três cenários
  variam o par e não a composição.

**Por fazer, antes do SMETANA:** harmonizar os identificadores do compartimento
extracelular entre o yeast-GEM e os três CarveFungi. O mapa da secção 3 resolve o
problema dentro deste notebook, ao traduzir nomes para ids, mas o SMETANA agrupa
o meio partilhado por string do id da troca.

## 9 · Bibliografia

1. Evers MS, Ramousse L, Morge C, et al. (2023). Thiamine and biotin: relevance in the production of volatile and non-volatile compounds during *Saccharomyces cerevisiae* alcoholic fermentation in synthetic grape must. *Foods* 12(5):972. — receita do MS300, secção 2.2.
2. Varela C, Pizarro F, Agosin E (2004). Biomass content governs fermentation rate in nitrogen-deficient wine musts. *Applied and Environmental Microbiology* 70(6):3392–3400. — µ = 0.2 h⁻¹ e X(t₂) = 5.8 gDW/L em mosto sintético a 300 mg N/L; 1.5 gDW/L a 50 mg N/L.
3. Brou P, Taillandier P, Beaufort S, Brandam C. Modelling of *S. cerevisiae* and *T. delbrueckii* pure culture fermentation in synthetic media using a compartmental nitrogen model. *OENO One*. — MS170 e MS300, biomassa máxima perto de 8 g/L nas duas espécies.
4. Sauer U, Lasko DR, Fiaux J, et al. (1999). Metabolic flux ratio analysis of genetic and environmental modulations of *Escherichia coli* central carbon metabolism. *Journal of Bacteriology* 181:6679–6688. — origem da relação de consumo específico em batch.
5. Crépin L, Nidelet T, Sanchez I, Dequin S, Camarasa C (2012). Sequential use of nitrogen compounds by *Saccharomyces cerevisiae* during wine fermentation. *Applied and Environmental Microbiology* 78:8102–8111. — ordem de grandeza das taxas específicas de consumo de aminoácidos em mosto.
6. Lu H, Li F, Sánchez BJ, et al. (2019). A consensus *S. cerevisiae* metabolic model Yeast8. *Nature Communications* 10:3586.
7. SysBioChalmers/yeast-GEM, `code/otherChanges/anaerobicModel.m` — protocolo anaeróbio: abre esteróis e ácidos gordos insaturados sem tecto derivado do meio.
8. Panozzo C, Nawara M, Suski C, et al. (2002). Aerobic and anaerobic NAD⁺ metabolism in *Saccharomyces cerevisiae*. *FEBS Letters* 517:97–102. — a via de novo do NAD a partir do triptofano depende de O₂, e é por isso que o nicotinato é essencial em anaerobiose.